# NLP – tekstvoorbewerking leesmateriaal

In dit notebook bewerken we de kolom `gecombineerde_tekst` voor met NLP-technieken, zodat de content-based recommender er later vectoren (TF-IDF) van kan maken.

| Stap | Resultaat |
|---|---|
| Data laden | df met `gecombineerde_tekst` |
| Lowercase | `tekst_lowercase` |
| Tekst opschonen | `tekst_schoon` |
| Tokenization | `tokens` |
| Stopwoorden verwijderen | `tokens_zonder_stopwoorden` |
| Stemming / lemmatization | `tokens_basisvorm` |
| Tekst samenvoegen | `tekst_nlp` |
| Resultaat controleren | vergelijking van 3 boeken |

## 0. Omgeving voorbereiden
NLTK downloadt de Nederlandse stopwoordenlijst en de tokenizer-modellen (eenmalig, internet nodig).

In [1]:
# !pip install nltk pandas
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nannalakens/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/nannalakens/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/nannalakens/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/nannalakens/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 1. Data laden
We laden de dataset en controleren of `gecombineerde_tekst` aanwezig is. Ontbreekt de kolom, dan maken we hem opnieuw aan uit dezelfde kolommen als in de data-preparation-stap.

In [2]:
df = pd.read_csv('../Docs/leesmateriaal.csv')

tekstkolommen = ['Korte omschrijving', 'Themas_boek', 'trefwoorden', 'onderwerp',
                 'genre', 'Type materiaal', 'Niveau', 'doelgroep']

if 'gecombineerde_tekst' not in df.columns:
    df['gecombineerde_tekst'] = df[tekstkolommen].fillna('').astype(str).agg(' '.join, axis=1)

print('Kolom aanwezig:', 'gecombineerde_tekst' in df.columns)
print('Aantal rijen:', len(df))
print('Lege teksten:', (df['gecombineerde_tekst'].str.strip() == '').sum())
df[['Titel', 'gecombineerde_tekst']].head()

Kolom aanwezig: True
Aantal rijen: 283
Lege teksten: 0


,Titel,gecombineerde_tekst
0,Het dagboek van Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...
1,Blauw Water,Spannende thriller waarin een vrouw wordt geco...
2,Boer vindt vrouw,"Toegankelijk verhaal over liefde, relaties en ..."
3,Cadeau voor u!,Bundel toegankelijke verhalen over verrassende...
4,De biecht,Verhaal waarin verborgen geheimen en persoonli...


In [35]:
df = pd.read_csv("../../lu2_datasets/Leescatalogus_gecombineerde_tekst.csv")
df["gecombineerde_tekst"]
print('Kolom aanwezig:', 'gecombineerde_tekst' in df.columns)
print('Aantal rijen:', len(df))
print('Lege teksten:', (df['gecombineerde_tekst'].str.strip() == '').sum())
df[['titel', 'gecombineerde_tekst']].head()

Kolom aanwezig: True
Aantal rijen: 283
Lege teksten: 0


,titel,gecombineerde_tekst
0,Het dagboek van Anne Frank,"Dagboek van een Joods meisje dat tijdens de Tweede Wereldoorlog onderduikt in Amsterdam. Haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. WOII;onderduik WOII;onderduik WOII fictie mbo 3F boek"
1,Blauw Water,"Spannende thriller waarin een vrouw wordt geconfronteerd met stalking en bedreiging. Terwijl de spanning oploopt, moet zij alles op alles zetten om zichzelf en haar gezin te beschermen. spanning;stalking spanning;stalking spanning fictie mbo/nt2 3F boek"
2,Boer vindt vrouw,"Toegankelijk verhaal over liefde, relaties en de zoektocht naar een passende partner. Het plattelandsleven vormt het decor voor herkenbare menselijke ervaringen en keuzes. liefde;relaties liefde;relaties liefde fictie mbo/nt2 2F boek"
3,Cadeau voor u!,"Bundel toegankelijke verhalen over verrassende ontmoetingen, kleine gebaren en menselijke verbondenheid. De verhalen laten zien hoe gewone gebeurtenissen een bijzondere betekenis kunnen krijgen. ontmoeting;verbondenheid ontmoeting;verbondenheid ontmoeting fictie mbo/nt2 3F boek"
4,De biecht,Verhaal waarin verborgen geheimen en persoonlijke keuzes centraal staan. Een bekentenis zet gebeurtenissen in gang die grote gevolgen hebben voor betrokkenen. geheimen;schuld geheimen;schuld geheimen fictie mbo/nt2 2F boek


## 2. Lowercase
Alle tekst naar kleine letters, zodat *Voetbal* en *voetbal* als hetzelfde woord worden gezien.

In [42]:
tekst_lowercase = df['gecombineerde_tekst'].str.lower()
tekst_lowercase

0                     dagboek van een joods meisje dat tijdens de tweede wereldoorlog onderduikt in amsterdam. haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. woii;onderduik woii;onderduik woii fictie mbo 3f boek
1                               spannende thriller waarin een vrouw wordt geconfronteerd met stalking en bedreiging. terwijl de spanning oploopt, moet zij alles op alles zetten om zichzelf en haar gezin te beschermen. spanning;stalking spanning;stalking spanning fictie mbo/nt2 3f boek
2                                                   toegankelijk verhaal over liefde, relaties en de zoektocht naar een passende partner. het plattelandsleven vormt het decor voor herkenbare menselijke ervaringen en keuzes. liefde;relaties liefde;relaties liefde fictie mbo/nt2 2f boek
3      bundel toegankelijke verhalen over verrassende ontmoetingen, kleine gebaren en menselijke verbondenheid. de verhalen laten zien hoe gew

## 3. Tekst opschonen
We verwijderen leestekens, cijfers en dubbele spaties. Letters met accenten (é, ë, ï) blijven behouden, omdat die in Nederlandse woorden voorkomen.

Let op: cijfers worden ook verwijderd. Leesniveaus als *A2* of *B1* worden daardoor *a* en *b*; het niveau gebruiken we daarom in het model als aparte filter/kenmerk en niet via de tekst.

In [37]:
def opschonen(tekst):
    tekst = re.sub(r'[^a-zà-ÿ0-9+\s]', ' ', tekst)   # alleen letters en spaties behouden
    tekst = re.sub(r'\s+', ' ', tekst)            # meerdere spaties -> één spatie
    return tekst.strip()

tekst_schoon = tekst_lowercase.apply(opschonen)
tekst_schoon

0                      dagboek van een joods meisje dat tijdens de tweede wereldoorlog onderduikt in amsterdam haar persoonlijke observaties geven een indringend beeld van angst hoop groei en het dagelijks leven onder vervolging woii onderduik woii onderduik woii fictie mbo 3f boek
1                               spannende thriller waarin een vrouw wordt geconfronteerd met stalking en bedreiging terwijl de spanning oploopt moet zij alles op alles zetten om zichzelf en haar gezin te beschermen spanning stalking spanning stalking spanning fictie mbo nt2 3f boek
2                                                   toegankelijk verhaal over liefde relaties en de zoektocht naar een passende partner het plattelandsleven vormt het decor voor herkenbare menselijke ervaringen en keuzes liefde relaties liefde relaties liefde fictie mbo nt2 2f boek
3      bundel toegankelijke verhalen over verrassende ontmoetingen kleine gebaren en menselijke verbondenheid de verhalen laten zien hoe gewone gebeurt

## 4. Tokenization
De tekst wordt gesplitst in losse woorden (tokens) met de Nederlandse tokenizer van NLTK.

In [38]:
tokens = tekst_schoon.apply(lambda t: word_tokenize(t, language='dutch'))
tokens

0                [dagboek, van, een, joods, meisje, dat, tijdens, de, tweede, wereldoorlog, onderduikt, in, amsterdam, haar, persoonlijke, observaties, geven, een, indringend, beeld, van, angst, hoop, groei, en, het, dagelijks, leven, onder, vervolging, woii, onderduik, woii, onderduik, woii, fictie, mbo, 3f, boek]
1                          [spannende, thriller, waarin, een, vrouw, wordt, geconfronteerd, met, stalking, en, bedreiging, terwijl, de, spanning, oploopt, moet, zij, alles, op, alles, zetten, om, zichzelf, en, haar, gezin, te, beschermen, spanning, stalking, spanning, stalking, spanning, fictie, mbo, nt2, 3f, boek]
2                                                   [toegankelijk, verhaal, over, liefde, relaties, en, de, zoektocht, naar, een, passende, partner, het, plattelandsleven, vormt, het, decor, voor, herkenbare, menselijke, ervaringen, en, keuzes, liefde, relaties, liefde, relaties, liefde, fictie, mbo, nt2, 2f, boek]
3      [bundel, toegankelijke, verhalen, over, ve

## 5. Stopwoorden verwijderen
Veelvoorkomende Nederlandse woorden (*de, het, een, en, van*) zeggen niets over de inhoud van een boek. We gebruiken de Nederlandse NLTK-lijst en verwijderen ook losse letters (bijv. overblijfselen als *a* uit *A2*).

In [39]:
stopwoorden_nl = set(stopwords.words('dutch'))
print('Aantal stopwoorden:', len(stopwoorden_nl))

tokens_zonder_stopwoorden = tokens.apply(
    lambda tokens: [w for w in tokens if w not in stopwoorden_nl and len(w) > 1]
)

voor = tokens.str.len().sum()
na = tokens_zonder_stopwoorden.str.len().sum()
print(f'Tokens voor: {voor}, na: {na} ({(voor - na) / voor:.0%} verwijderd)')
tokens_zonder_stopwoorden

Aantal stopwoorden: 101
Tokens voor: 6115, na: 4663 (24% verwijderd)


0                                       [dagboek, joods, meisje, tijdens, tweede, wereldoorlog, onderduikt, amsterdam, persoonlijke, observaties, geven, indringend, beeld, angst, hoop, groei, dagelijks, leven, vervolging, woii, onderduik, woii, onderduik, woii, fictie, mbo, 3f, boek]
1                                                                  [spannende, thriller, waarin, vrouw, geconfronteerd, stalking, bedreiging, terwijl, spanning, oploopt, zetten, zichzelf, gezin, beschermen, spanning, stalking, spanning, stalking, spanning, fictie, mbo, nt2, 3f, boek]
2                                                                [toegankelijk, verhaal, liefde, relaties, zoektocht, passende, partner, plattelandsleven, vormt, decor, herkenbare, menselijke, ervaringen, keuzes, liefde, relaties, liefde, relaties, liefde, fictie, mbo, nt2, 2f, boek]
3      [bundel, toegankelijke, verhalen, verrassende, ontmoetingen, kleine, gebaren, menselijke, verbondenheid, verhalen, laten, zien, gewone, ge

## 6. Stemming / lemmatization
- **Lemmatization** brengt een woord terug naar zijn woordenboekvorm (*liep → lopen*). De WordNet-lemmatizer uit NLTK werkt alleen voor Engels; voor Nederlands is een extra taalmodel nodig (bijv. spaCy `nl_core_news_sm`).
- **Stemming** knipt woorden af tot hun stam (*vriendschappen → vriendschap*). De Snowball-stemmer van NLTK ondersteunt Nederlands.

**Keuze:** we gebruiken de Nederlandse **SnowballStemmer**. Die is snel, heeft geen extra model nodig en zorgt ervoor dat varianten van hetzelfde woord samenvallen, wat de matching tussen leesprofiel en boek verbetert. Nadeel: de stam is soms geen bestaand woord (*avontuur → avontur*), maar voor het berekenen van overeenkomst is dat geen probleem.

In [40]:
stemmer = SnowballStemmer('dutch')

voorbeelden = ['vriendschappen', 'vriendschap', 'avonturen', 'avontuur', 'oorlogen', 'spannend', 'spannende']
for w in voorbeelden:
    print(f'{w:15} -> {stemmer.stem(w)}')

vriendschappen  -> vriendschapp
vriendschap     -> vriendschap
avonturen       -> avontur
avontuur        -> avontur
oorlogen        -> oorlog
spannend        -> spannend
spannende       -> spannend


In [41]:
tokens_basisvorm = tokens_zonder_stopwoorden.apply(
    lambda tokens: [stemmer.stem(w) for w in tokens]
)

uniek_voor = len(set(w for t in tokens_zonder_stopwoorden for w in t))
uniek_na = len(set(w for t in tokens_basisvorm for w in t))
print(f'Unieke woorden voor stemming: {uniek_voor}, na stemming: {uniek_na}')
tokens_basisvorm

Unieke woorden voor stemming: 836, na stemming: 758


0      [dagboek, jod, meisj, tijden, twed, wereldoorlog, onderduikt, amsterdam, person, observaties, gev, indring, beeld, angst, hop, groei, dagelijk, lev, vervolg, woii, onderduik, woii, onderduik, woii, fictie, mbo, 3f, boek]
1                  [spannend, thriller, waarin, vrouw, geconfronteerd, stalking, bedreig, terwijl, spanning, oploopt, zet, zichzelf, gezin, bescherm, spanning, stalking, spanning, stalking, spanning, fictie, mbo, nt2, 3f, boek]
2                              [toegank, verhal, liefd, relaties, zoektocht, passend, partner, plattelandslev, vormt, decor, herken, menselijk, ervar, keuzes, liefd, relaties, liefd, relaties, liefd, fictie, mbo, nt2, 2f, boek]
3       [bundel, toegank, verhal, verrass, ontmoet, klein, gebar, menselijk, verbond, verhal, lat, zien, gewon, gebeurteniss, bijzonder, betekenis, krijg, ontmoet, verbond, ontmoet, verbond, ontmoet, fictie, mbo, nt2, 3f, boek]
4                                [verhal, waarin, verborg, geheim, person, keuzes, centr

## 7. Tekst samenvoegen
De bewerkte tokens worden weer één tekst. Deze kolom `tekst_nlp` is de input voor de TF-IDF-vectorizer in het model.

In [43]:
df['tekst_nlp']= tokens_basisvorm.apply(' '.join)
df['tekst_nlp']

0      dagboek jod meisj tijden twed wereldoorlog onderduikt amsterdam person observaties gev indring beeld angst hop groei dagelijk lev vervolg woii onderduik woii onderduik woii fictie mbo 3f boek
1              spannend thriller waarin vrouw geconfronteerd stalking bedreig terwijl spanning oploopt zet zichzelf gezin bescherm spanning stalking spanning stalking spanning fictie mbo nt2 3f boek
2                          toegank verhal liefd relaties zoektocht passend partner plattelandslev vormt decor herken menselijk ervar keuzes liefd relaties liefd relaties liefd fictie mbo nt2 2f boek
3      bundel toegank verhal verrass ontmoet klein gebar menselijk verbond verhal lat zien gewon gebeurteniss bijzonder betekenis krijg ontmoet verbond ontmoet verbond ontmoet fictie mbo nt2 3f boek
4                             verhal waarin verborg geheim person keuzes central stan bekentenis zet gebeurteniss gang grot gevolg betrokken geheim schuld geheim schuld geheim fictie mbo nt2 2f boek
     

## 8. Resultaat controleren
We vergelijken de oorspronkelijke en de verwerkte tekst van drie boeken.

In [44]:
pd.set_option('display.max_colwidth', None)
for _, rij in df.sample(3, random_state=42).iterrows():
    print('=' * 80)
    print('TITEL     :', rij['titel'])
    print('ORIGINEEL :', rij['gecombineerde_tekst'])
    print('VERWERKT  :', rij['tekst_nlp'])
    print(f"Woorden: {len(rij['gecombineerde_tekst'].split())} -> {len(rij['tekst_nlp'])}")

TITEL     : De val
ORIGINEEL : Indringende novelle over de gevolgen van oorlog, verlies en herinnering. Het verleden blijft doorwerken in het leven van de personages. oorlog;verlies oorlog;verlies oorlog fictie mbo 3F boek
VERWERKT  : indring novell gevolg oorlog verlies herinner verled blijft doorwerk lev personages oorlog verlies oorlog verlies oorlog fictie mbo 3f boek
Woorden: 27 -> 139
TITEL     : De brief van Jelle
ORIGINEEL : Youngadultthriller waarin geheimen en macht centraal staan. Thema's als macht en doorzetten komen op toegankelijke wijze aan bod. geheimen;macht geheimen;macht geheimen fictie;thriller mbo/nt2 2F boek
VERWERKT  : youngadultthriller waarin geheim macht central stan thema macht doorzet kom toegank wijz bod geheim macht geheim macht geheim fictie thriller mbo nt2 2f boek
Woorden: 25 -> 157
TITEL     : Vanaf 2 augustus wordt duidelijker of het AI is of echt
ORIGINEEL : AI, privacy, deepfakes informatief mbo 3F artikel
VERWERKT  : ai privacy deepfakes informatie

In [49]:
pd.set_option('display.max_colwidth', None)
vergelijking = pd.DataFrame({
    "Originele tekst": df["gecombineerde_tekst"],
    "Na NLP": df["tekst_nlp"]
})

display(vergelijking.head(3))

,Originele tekst,Na NLP
0,"Dagboek van een Joods meisje dat tijdens de Tweede Wereldoorlog onderduikt in Amsterdam. Haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. WOII;onderduik WOII;onderduik WOII fictie mbo 3F boek",dagboek jod meisj tijden twed wereldoorlog onderduikt amsterdam person observaties gev indring beeld angst hop groei dagelijk lev vervolg woii onderduik woii onderduik woii fictie mbo 3f boek
1,"Spannende thriller waarin een vrouw wordt geconfronteerd met stalking en bedreiging. Terwijl de spanning oploopt, moet zij alles op alles zetten om zichzelf en haar gezin te beschermen. spanning;stalking spanning;stalking spanning fictie mbo/nt2 3F boek",spannend thriller waarin vrouw geconfronteerd stalking bedreig terwijl spanning oploopt zet zichzelf gezin bescherm spanning stalking spanning stalking spanning fictie mbo nt2 3f boek
2,"Toegankelijk verhaal over liefde, relaties en de zoektocht naar een passende partner. Het plattelandsleven vormt het decor voor herkenbare menselijke ervaringen en keuzes. liefde;relaties liefde;relaties liefde fictie mbo/nt2 2F boek",toegank verhal liefd relaties zoektocht passend partner plattelandslev vormt decor herken menselijk ervar keuzes liefd relaties liefd relaties liefd fictie mbo nt2 2f boek


## 9. Opslaan
We slaan het resultaat op zodat het modelnotebook het kan gebruiken.

In [34]:
df.to_csv('../../lu2_datasets/Leescatalogus_nlp.csv', index=False)
print('Opgeslagen:', df.shape)

Opgeslagen: (283, 31)
